In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
def remap_mask(mask):

    masking = mask.long()

    unique_values = torch.unique(masking)

    remapped_mask = torch.zeros_like(masking)

    for new_val , old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[masking == old_val] = new_val

    return remapped_mask

In [ ]:
# TO DO
import glob
import torch
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms
import numpy as np


data_root = path
if len(os.listdir(path)) == 1 and os.path.isdir(os.path.join(path, os.listdir(path)[0])):
    data_root = os.path.join(path, os.listdir(path)[0])


# 2
class UnderwaterDataset(Dataset):
    def __init__(self, root_dir, transform=None, target_transform=None):


        self.image_paths = sorted(glob.glob(os.path.join(root_dir, "**", "images", "*.jpg"), recursive=True))
        self.mask_paths = sorted(glob.glob(os.path.join(root_dir, "**", "masks", "*.png"), recursive=True))

        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # نسوي mask
        image = Image.open(self.image_paths[idx]).convert("RGB")
        mask = Image.open(self.mask_paths[idx]).convert("L")

        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            mask = self.target_transform(mask)


        mask = torch.as_tensor(np.array(mask), dtype=torch.long)
        mask = remap_mask(mask)

        return image, mask

# 3. التحويلات
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

target_transform = transforms.Compose([
    transforms.Resize((256, 256), interpolation=Image.NEAREST),
])

dataset = UnderwaterDataset(root_dir=data_root, transform=transform, target_transform=target_transform)



if len(dataset) > 0:
    dataloader = DataLoader(dataset, batch_size=8, shuffle=True)


    import matplotlib.pyplot as plt
    img, msk = dataset[0]
    plt.subplot(1, 2, 1); plt.imshow(img.permute(1,2,0)); plt.title("Image")
    plt.subplot(1, 2, 2); plt.imshow(msk, cmap='jet'); plt.title("Mask")
    plt.show()
else:
    print(" i can not find image(jpg or png).")

In [ ]:
# TO DO

!pip install -q segmentation_models_pytorch

import segmentation_models_pytorch as smp
import torch


num_classes = 8


model = smp.Unet(
    encoder_name="efficientnet-b1",
    encoder_weights="imagenet",
    in_channels=3,
    classes=num_classes
)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"Model : {device}")

In [ ]:
# TO DO
from tqdm import tqdm
import torch



def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0


    pbar = tqdm(dataloader, desc="Training")
    for images, masks in pbar:
        images, masks = images.to(device), masks.to(device)

        # Forward
        outputs = model(images)

        loss = criterion(outputs, masks)

        # Backward
        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        pbar.set_postfix(loss=loss.item())

    return running_loss / len(dataloader)


def dice_coefficient(preds, targets, num_classes=8):
    smooth = 1.0
    dice = 0

    preds = torch.softmax(preds, dim=1).argmax(dim=1)

    for cl in range(num_classes):

        p = (preds == cl).float()

        t = (targets == cl).float()

        intersection = (p * t).sum()
        dice += (2. * intersection + smooth) / (p.sum() + t.sum() + smooth)

    return dice / num_classes


def validate(model, dataloader, criterion, device, num_classes=8):
    model.eval()
    running_loss = 0.0
    total_dice = 0.0

    with torch.no_grad():
        pbar = tqdm(dataloader, desc="Validating")
        for images, masks in pbar:
            images, masks = images.to(device), masks.to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)
            running_loss += loss.item()


            dice = dice_coefficient(outputs, masks, num_classes=num_classes)
            total_dice += dice.item()

            pbar.set_postfix(loss=loss.item(), dice=dice.item())

    avg_loss = running_loss / len(dataloader)
    avg_dice = total_dice / len(dataloader)
    return avg_loss, avg_dice

In [ ]:
# TO DO
import matplotlib.pyplot as plt
import torch.optim as optim
import torch.nn as nn



criterion = nn.CrossEntropyLoss()


optimizer = optim.Adam(model.parameters(), lr=0.001)


num_epochs = 10
train_losses = []
val_losses = []
val_dices = []

print("Start Train")

for epoch in range(num_epochs):

    train_loss = train_one_epoch(model, dataloader, criterion, optimizer, device)
    val_loss, val_dice = validate(model, dataloader, criterion, device, num_classes=8)


    train_losses.append(train_loss)

    val_losses.append(val_loss)

    val_dices.append(val_dice)


    print(f"Epoch [{epoch+1}/{num_epochs}] - "
          f"Train Loss: {train_loss:.4f}, "
          f"Val Loss: {val_loss:.4f}, "
          f"Val Dice Coefficient: {val_dice:.4f}")


plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Training Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Training and Validation Loss Curve')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# TO DO
import matplotlib.pyplot as plt
import torch

def vis_predict(model, dataset, device, num_samples=3):
    model.eval()
    fig, axes = plt.subplots(num_samples, 3, figsize=(15, num_samples * 5))

    with torch.no_grad():


        for i in range(num_samples):
            # 1. mask
            image, mask = dataset[i]

            # 2.
            input_image = image.unsqueeze(0).to(device)

            # 3.
            output = model(input_image)

            # 4.
            pred_mask = torch.argmax (output, dim=1).cpu().squeeze().numpy()

            # 5.
            img_display = image.permute(1, 2, 0) .numpy()
            img_display = np.clip(img_display * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406], 0, 1)


            axes[i, 0].imshow(img_display)
            axes[i, 0].set_title("occuin")
            axes[i, 0].axis('off')


            axes[i, 1].imshow(mask.numpy(), cmap='jet')
            axes[i, 1].set_title("Ground")
            axes[i, 1].axis('off')


            axes[i, 2].imshow(pred_mask, cmap='jet')
            axes[i, 2].set_title("Model Prediction")
            axes[i, 2].axis('off')

    plt.tight_layout()
    plt.show()


vis_predict(model, dataset, device, num_samples=3)